# 🎬 AI-Powered Video Remediation Module
**Automatically detect and remediate harmful content in videos through selective blurring and timestamp-based audio censoring.**

---
### Pipeline Overview
```
Upload Video → Frame Extraction → Object/Nudity/Violence Detection → Blur Harmful Regions
            → Audio Extraction → Speech Transcription → Detect Harmful Speech → Mute/Beep
            → Merge Final Video → Generate JSON Report
```

### What this notebook does:
- **Visual Detection**: YOLO-based object detection (weapons, cigarettes, alcohol) + NudeNet for nudity
- **Audio Detection**: Whisper transcription + profanity/toxicity classification
- **Remediation**: Selective blurring of harmful regions + muting/beeping harmful speech
- **Output**: Safe remediated video + timestamped JSON report

## 1. 📦 Install Dependencies

In [1]:
# Install all required packages
%pip install ultralytics nudenet openai-whisper better-profanity detoxify opencv-python-headless ffmpeg-python numpy tqdm

# System dependency: ffmpeg (required for audio processing)
# On Ubuntu/Debian: sudo apt-get install -y ffmpeg
# On macOS: brew install ffmpeg
# On Windows: download from https://ffmpeg.org/download.html

import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ ffmpeg is available')
else:
    print('❌ ffmpeg not found. Please install it before proceeding.')

  Using cached ultralytics-8.4.42-py3-none-any.whl.metadata (39 kB)
  Using cached nudenet-3.4.2-py3-none-any.whl.metadata (4.3 kB)
  Using cached openai_whisper-20250625.tar.gz (803 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached better_profanity-0.7.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached detoxify-0.5.2-py3-none-any.whl.metadata (13 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached ffmpeg_python-0.2.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 973.8 kB/s eta 0:00:00 0:00:01
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached scipy-1.

## 2. 🔧 Imports & Configuration

In [66]:
import cv2
import json
import os
import tempfile
import subprocess
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ─── Configuration ────────────────────────────────────────────────────────────

CONFIG = {
    # Input / Output
    "input_video":  "/home/atharva-manchalkar/Downloads/project/fu.mp4",      # ← change to your video path
    "output_video": "output_safe.mp4",
    "report_path":  "remediation_report.json",

    # Visual detection
    "frame_interval_sec": 0.5,              # analyze every N seconds (overridden by analyze_every_frame)
    "analyze_every_frame": True,            # set True to analyze every video frame (slower, more accurate)
    "yolo_model":   "/home/atharva-manchalkar/Downloads/project/best.pt",           # or path to custom best.pt
    "yolo_confidence": 0.45,
    "blur_strength": 51,                    # Gaussian blur kernel (odd number)

    # Harmful YOLO classes to detect (COCO class names)
    "harmful_visual_classes": [
        "knife", "scissors",               # bladed weapons
        "gun", "pistol", "rifle",          # firearms (custom model classes)
        "bottle",                           # alcohol bottles
        "cigarette",                        # smoking (custom model)
        "weapon",                           # catch-all label from some models
    ],

    # Nudity detection
    "nudenet_labels": [
        "EXPOSED_ANUS", "EXPOSED_ARMPITS",
        "EXPOSED_BELLY", "EXPOSED_BUTTOCKS",
        "EXPOSED_BREAST_F", "EXPOSED_GENITALIA_F", "EXPOSED_GENITALIA_M",
    ],
    "nudenet_confidence": 0.5,

    # Temporal smoothing & tracking
    "smoothing_duration_sec": 2.0,          # longer: keep blur alive longer (was 0.5)
    "centroid_distance_threshold": 100,     # max distance in pixels to match same object across frames

    # Audio
    "whisper_model": "base",               # tiny | base | small | medium | large
    "beep_frequency_hz": 1000,
    "use_beep": True,                       # True = beep tone | False = silence

    # Profanity wordlist (add your own)
    "extra_bad_words": [],

    # Debugging — set True to print detection details per analysis tick
    "debug": False,  # Turn off for cleaner runs
}

print("✅ Configuration loaded")
print(f"   Input  : {CONFIG['input_video']}")
print(f"   Output : {CONFIG['output_video']}")
print(f"   Report : {CONFIG['report_path']}")
print(f"   Smoothing duration: {CONFIG['smoothing_duration_sec']}s")

✅ Configuration loaded
   Input  : /home/atharva-manchalkar/Downloads/project/fu.mp4
   Output : output_safe.mp4
   Report : remediation_report.json
   Smoothing duration: 2.0s


## 3. 🧠 Load Models

In [56]:
from ultralytics import YOLO
from nudenet import NudeDetector
import whisper
from better_profanity import profanity
from detoxify import Detoxify

print("Loading models — this may take a minute on first run...")

# YOLO — object detector
yolo_model = YOLO(CONFIG["yolo_model"])
print("✅ YOLO loaded")

# NudeNet — nudity/explicit content detector
nude_detector = NudeDetector()
print("✅ NudeNet loaded")

# Whisper — speech transcription with timestamps
whisper_model = whisper.load_model(CONFIG["whisper_model"])
print("✅ Whisper loaded")

# better-profanity — keyword-based profanity filter
profanity.load_censor_words()
if CONFIG["extra_bad_words"]:
    profanity.add_censor_words(CONFIG["extra_bad_words"])
print("✅ Profanity filter loaded")

# Detoxify — ML-based toxicity classifier
toxicity_model = Detoxify('original')
print("✅ Detoxify toxicity model loaded")

print("\n🎉 All models ready!")

Loading models — this may take a minute on first run...
✅ YOLO loaded
✅ NudeNet loaded
✅ Whisper loaded
✅ Profanity filter loaded


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 25234.37it/s]


✅ Detoxify toxicity model loaded

🎉 All models ready!


## 4. 🔍 Visual Detection — Frames

In [57]:
def is_harmful_class(class_name: str) -> bool:
    """Check if a detected class name is in the harmful list."""
    name = class_name.lower()
    return any(h in name for h in CONFIG["harmful_visual_classes"])



def blur_region(frame: np.ndarray, x1: int, y1: int, x2: int, y2: int,
                strength: int = 51) -> np.ndarray:
    """Apply Gaussian blur to a rectangular region of a frame."""
    h, w = frame.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return frame
    k = strength if strength % 2 == 1 else strength + 1  # must be odd
    roi = frame[y1:y2, x1:x2]
    frame[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (k, k), 0)
    return frame



def detect_and_remediate_frame(frame: np.ndarray, timestamp: float) -> tuple[np.ndarray, list]:
    """Run YOLO + NudeNet on a single frame; blur harmful regions; return detections.

    This function RETURNS detections but DOES NOT append to `video_flags` —
    per-frame logging is handled in `process_video_frames` so we can record one
    flag per blurred frame (avoids duplicate counts).
    """
    remediated = frame.copy()

    # ── YOLO object detection ─────────────────────────────────────────────
    yolo_results = yolo_model(frame, conf=CONFIG["yolo_confidence"], verbose=False)

    # Collect raw detections for debug output and return
    detections = []
    for result in yolo_results:
        for box in result.boxes:
            cls_id = int(box.cls[0])
            cls_name = yolo_model.names.get(cls_id, str(cls_id))
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            det = {
                "class": cls_name,
                "confidence": round(conf, 3),
                "bbox": [x1, y1, x2, y2],
            }
            detections.append(det)

            # If harmful, apply blur here (but DO NOT append to flags list)
            if is_harmful_class(cls_name):
                remediated = blur_region(remediated, x1, y1, x2, y2,
                                         CONFIG["blur_strength"])

    if CONFIG.get("debug"):
        print(f"[DEBUG] {timestamp:.3f}s — YOLO raw detections: {detections}")

    # ── NudeNet nudity detection ──────────────────────────────────────────
    _, tmp_path = tempfile.mkstemp(suffix='.jpg')
    try:
        cv2.imwrite(tmp_path, frame)
        n_detections = nude_detector.detect(tmp_path)
        for det in n_detections:
            label = det.get('class', '')
            score = det.get('score', 0)
            if label in CONFIG["nudenet_labels"] and score >= CONFIG["nudenet_confidence"]:
                box = det.get('box', [])
                if len(box) == 4:
                    x1, y1, x2, y2 = map(int, box)
                    detections.append({
                        "class": label,
                        "confidence": round(score, 3),
                        "bbox": [x1, y1, x2, y2],
                    })
                    remediated = blur_region(remediated, x1, y1, x2, y2,
                                             CONFIG["blur_strength"])
    finally:
        os.unlink(tmp_path)

    return remediated, detections


print("✅ Visual detection helpers defined")

✅ Visual detection helpers defined


## 5. 🎤 Audio Detection & Remediation

In [58]:
TOXICITY_THRESHOLD = 0.6  # Detoxify score threshold for flagging


def extract_audio(video_path: str, audio_path: str) -> bool:
    """Extract audio track from video to a WAV file using ffmpeg."""
    cmd = [
        'ffmpeg', '-y', '-i', video_path,
        '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1',
        audio_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0


def transcribe_audio(audio_path: str) -> list[dict]:
    """Transcribe audio using Whisper; return list of word-level segments."""
    result = whisper_model.transcribe(
        audio_path,
        word_timestamps=True,
        language=None,          # auto-detect language
        verbose=False,
    )
    segments = []
    for seg in result.get('segments', []):
        words = seg.get('words', [])
        if words:
            for w in words:
                segments.append({
                    'start': round(w['start'], 3),
                    'end':   round(w['end'],   3),
                    'text':  w['word'].strip(),
                })
        else:
            # fallback: no word-level timestamps
            segments.append({
                'start': round(seg['start'], 3),
                'end':   round(seg['end'],   3),
                'text':  seg['text'].strip(),
            })
    return segments


def is_harmful_speech(text: str) -> tuple[bool, str]:
    """Return (is_harmful, reason) for a speech segment."""
    # 1. Keyword-based profanity check
    if profanity.contains_profanity(text):
        return True, "profanity"

    # 2. ML-based toxicity classification (skip very short words)
    if len(text.split()) >= 2:
        scores = toxicity_model.predict(text)
        toxic_labels = [
            'toxicity', 'severe_toxicity', 'obscene',
            'threat', 'insult', 'identity_attack', 'sexual_explicit'
        ]
        for label in toxic_labels:
            if scores.get(label, 0) >= TOXICITY_THRESHOLD:
                return True, label

    return False, ""


def detect_harmful_segments(segments: list[dict]) -> list[dict]:
    """Flag harmful speech segments and return audio_flags list."""
    audio_flags = []
    # Group consecutive harmful words into spans for smoother muting
    current_flag = None

    for seg in segments:
        harmful, reason = is_harmful_speech(seg['text'])
        if harmful:
            if current_flag and seg['start'] - current_flag['end'] < 0.3:
                # Extend current flag
                current_flag['end']  = seg['end']
                current_flag['text'] += ' ' + seg['text']
            else:
                if current_flag:
                    audio_flags.append(current_flag)
                current_flag = {
                    'start':  seg['start'],
                    'end':    seg['end'],
                    'text':   seg['text'],
                    'reason': reason,
                    'action': 'beeped' if CONFIG['use_beep'] else 'muted',
                }
        else:
            if current_flag:
                audio_flags.append(current_flag)
                current_flag = None

    if current_flag:
        audio_flags.append(current_flag)

    return audio_flags


print("✅ Audio detection helpers defined")

✅ Audio detection helpers defined


## 6. 🔊 Audio Remediation (Mute / Beep)

In [59]:
import math
import wave
import struct


def generate_beep(duration_sec: float, sample_rate: int = 16000,
                  freq_hz: int = 1000, amplitude: float = 0.3) -> bytes:
    """Generate raw PCM beep tone samples."""
    num_samples = int(sample_rate * duration_sec)
    samples = [
        int(amplitude * 32767 * math.sin(2 * math.pi * freq_hz * i / sample_rate))
        for i in range(num_samples)
    ]
    return struct.pack(f'{num_samples}h', *samples)


def remediate_audio(original_audio: str, flagged_segments: list[dict],
                    output_audio: str, use_beep: bool = True) -> None:
    """Read WAV, mute or beep flagged segments, write new WAV."""
    with wave.open(original_audio, 'rb') as wf:
        params      = wf.getparams()
        sample_rate = wf.getframerate()
        n_channels  = wf.getnchannels()
        sampwidth   = wf.getsampwidth()
        n_frames    = wf.getnframes()
        raw_data    = bytearray(wf.readframes(n_frames))

    bytes_per_sample = sampwidth * n_channels

    for seg in flagged_segments:
        start_frame = int(seg['start'] * sample_rate)
        end_frame   = int(seg['end']   * sample_rate)
        start_byte  = start_frame * bytes_per_sample
        end_byte    = min(end_frame * bytes_per_sample, len(raw_data))
        duration    = seg['end'] - seg['start']

        if use_beep:
            beep_pcm = generate_beep(
                duration, sample_rate,
                CONFIG['beep_frequency_hz']
            )
            beep_len = min(len(beep_pcm), end_byte - start_byte)
            raw_data[start_byte:start_byte + beep_len] = beep_pcm[:beep_len]
        else:
            # Silence
            raw_data[start_byte:end_byte] = bytes(end_byte - start_byte)

    with wave.open(output_audio, 'wb') as wf:
        wf.setparams(params)
        wf.writeframes(bytes(raw_data))


print("✅ Audio remediation helpers defined")

✅ Audio remediation helpers defined


## 7. 🎥 Video Processing Pipeline

In [60]:
def process_video_frames(video_path: str, frames_output: str) -> tuple[list, list, dict]:
    """
    Read video with ROBUST CENTROID-BASED TRACKING.
    - Detect weapons on sampled frames
    - Match detections to tracked objects using centroid distance
    - Interpolate bbox position across frames for smooth following
    - Keep blur alive for `smoothing_duration_sec` after last detection
    
    Returns (video_flags, video_meta).
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps       = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    meta = {"fps": fps, "width": width, "height": height, "total_frames": total_frames}
    print(f"📹 Video: {width}x{height} @ {fps:.1f} fps | {total_frames} frames")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(frames_output, fourcc, fps, (width, height))

    video_flags = []
    
    # Determine analysis interval
    if CONFIG.get("analyze_every_frame"):
        interval_frames = 1
    else:
        interval_frames = max(1, int(fps * CONFIG["frame_interval_sec"]))

    # Temporal smoothing window (frames)
    smoothing_frames = max(1, int(fps * CONFIG.get('smoothing_duration_sec', 2.0)))
    centroid_threshold = CONFIG.get('centroid_distance_threshold', 100)

    # Tracked objects: {id: {bbox, class, confidence, last_seen_frame, centroid_history}}
    # centroid_history is list of (frame_idx, centroid_x, centroid_y) for interpolation
    tracked_objects = {}
    next_track_id = 0

    def get_centroid(bbox):
        """Get center point of bounding box."""
        x1, y1, x2, y2 = bbox
        return ((x1 + x2) // 2, (y1 + y2) // 2)

    def interpolate_bbox(bbox1, bbox2, alpha):
        """Linear interpolation between two bboxes."""
        x1a, y1a, x2a, y2a = bbox1
        x1b, y1b, x2b, y2b = bbox2
        x1 = int(x1a + alpha * (x1b - x1a))
        y1 = int(y1a + alpha * (y1b - y1a))
        x2 = int(x2a + alpha * (x2b - x2a))
        y2 = int(y2a + alpha * (y2b - y2a))
        return [x1, y1, x2, y2]

    frame_idx = 0
    last_remediated = None
    blurred_frames_count = 0

    pbar = tqdm(total=total_frames, desc="Processing frames", unit="frame")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        timestamp = frame_idx / fps

        # Prune expired tracked objects
        tracked_objects = {
            k: v for k, v in tracked_objects.items()
            if frame_idx - v['last_seen_frame'] <= smoothing_frames
        }

        # Run detection on sampled frames
        if frame_idx % interval_frames == 0:
            remediated, detections = detect_and_remediate_frame(frame, timestamp)

            # Match detections to tracked objects using centroid distance
            used_detections = set()
            for track_id, track in tracked_objects.items():
                track_centroid = get_centroid(track['bbox'])
                best_match_idx = None
                best_dist = centroid_threshold

                for det_idx, det in enumerate(detections):
                    if det_idx in used_detections:
                        continue
                    if not is_harmful_class(det['class']):
                        continue
                    det_centroid = get_centroid(det['bbox'])
                    dist = np.sqrt((track_centroid[0] - det_centroid[0])**2 + 
                                   (track_centroid[1] - det_centroid[1])**2)
                    if dist < best_dist:
                        best_dist = dist
                        best_match_idx = det_idx

                if best_match_idx is not None:
                    # Update tracked object
                    det = detections[best_match_idx]
                    track['bbox'] = det['bbox']
                    track['confidence'] = det['confidence']
                    track['class'] = det['class']
                    track['last_seen_frame'] = frame_idx
                    track['centroid_history'].append((frame_idx, get_centroid(det['bbox'])))
                    used_detections.add(best_match_idx)

            # Create new tracks for unmatched detections
            for det_idx, det in enumerate(detections):
                if det_idx not in used_detections and is_harmful_class(det['class']):
                    tracked_objects[next_track_id] = {
                        'bbox': det['bbox'],
                        'class': det['class'],
                        'confidence': det['confidence'],
                        'last_seen_frame': frame_idx,
                        'centroid_history': [(frame_idx, get_centroid(det['bbox']))],
                    }
                    next_track_id += 1

            last_remediated = remediated
        else:
            # Interpolate bbox for intermediate frames
            rem = frame.copy()

            for track in tracked_objects.values():
                last_seen = track['last_seen_frame']
                if len(track['centroid_history']) >= 1:
                    # Linear interpolation between last detection and next
                    # For now, use last known bbox (could improve with prediction)
                    x1, y1, x2, y2 = track['bbox']
                    rem = blur_region(rem, x1, y1, x2, y2, CONFIG['blur_strength'])

            last_remediated = rem

        # Write frame and log if blurred
        writer.write(last_remediated)
        
        if tracked_objects:
            blurred_frames_count += 1
            if frame_idx % interval_frames == 0:  # Log only on detection frames
                video_flags.append({
                    'timestamp': round(timestamp, 3),
                    'types': list({t['class'] for t in tracked_objects.values()}),
                    'bboxes': [t['bbox'] for t in tracked_objects.values()],
                    'confidence': max([t['confidence'] for t in tracked_objects.values()] + [0]),
                    'action': 'blurred_frame',
                })

        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    writer.release()

    print(f"✅ Frame processing complete | total_frames={frame_idx} | blurred_frames={blurred_frames_count} | unique_visual_flags={len(video_flags)}")
    return video_flags, meta


print("✅ Frame processing pipeline defined")

✅ Frame processing pipeline defined


## 8. 🚀 Run the Full Pipeline

In [71]:
def run_pipeline(config: dict = CONFIG) -> dict:
    """
    Full end-to-end video remediation pipeline.
    Returns the JSON report dict.
    """
    input_path  = config["input_video"]
    output_path = config["output_video"]
    report_path = config["report_path"]

    if not os.path.exists(input_path):
        raise FileNotFoundError(
            f"Input video not found: {input_path}\n",
            "Update CONFIG['input_video'] to point to your video file."
        )

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_frames_video = os.path.join(tmp_dir, "frames_only.mp4")
        tmp_audio_orig   = os.path.join(tmp_dir, "audio_orig.wav")
        tmp_audio_clean  = os.path.join(tmp_dir, "audio_clean.wav")

        # ── Step 1: Visual remediation ────────────────────────────────────
        print("\n🎥 STEP 1 / 4 — Visual detection & remediation")
        video_flags, meta = process_video_frames(input_path, tmp_frames_video)

        # ── Step 2: Audio extraction ──────────────────────────────────────
        print("\n🎤 STEP 2 / 4 — Audio extraction")
        has_audio = extract_audio(input_path, tmp_audio_orig)
        audio_flags = []

        if has_audio and os.path.exists(tmp_audio_orig):
            # ── Step 3: Audio detection & remediation ─────────────────────
            print("\n🔍 STEP 3 / 4 — Speech transcription & harmful speech detection")
            segments    = transcribe_audio(tmp_audio_orig)
            audio_flags = detect_harmful_segments(segments)
            print(f"   Transcribed {len(segments)} word segment(s) | {len(audio_flags)} audio flag(s)")

            remediate_audio(tmp_audio_orig, audio_flags, tmp_audio_clean,
                            use_beep=config["use_beep"])
            clean_audio = tmp_audio_clean
        else:
            print("   ⚠️  No audio track found — skipping audio remediation.")
            clean_audio = None

        # ── Step 4: Merge video + audio ───────────────────────────────────
        print("\n🔗 STEP 4 / 4 — Merging remediated video & audio")
        if clean_audio:
            merge_cmd = [
                'ffmpeg', '-y',
                '-i', tmp_frames_video,
                '-i', clean_audio,
                '-c:v', 'libx264', '-crf', '23', '-preset', 'fast',
                '-c:a', 'aac', '-b:a', '128k',
                '-shortest',
                output_path,
            ]
        else:
            merge_cmd = [
                'ffmpeg', '-y',
                '-i', tmp_frames_video,
                '-c:v', 'libx264', '-crf', '23', '-preset', 'fast',
                output_path,
            ]

        merge_result = subprocess.run(merge_cmd, capture_output=True, text=True)
        if merge_result.returncode != 0:
            raise RuntimeError(f"ffmpeg merge failed:\n{merge_result.stderr}")

    # ── Generate JSON report ──────────────────────────────────────────────
    # Visual types: collect from either 'type' or 'types' keys
    visual_types_set = set()
    for f in video_flags:
        if 'type' in f:
            visual_types_set.add(f['type'])
        elif 'types' in f:
            for t in f['types']:
                visual_types_set.add(t)

    audio_types_set = set(f.get('reason') for f in audio_flags if 'reason' in f)

    report = {
        "input_video":   input_path,
        "output_video":  output_path,
        "video_meta":    meta,
        "video_flags":   video_flags,
        "audio_flags":   audio_flags,
        "summary": {
            "total_visual_flags": len(video_flags),
            "total_audio_flags":  len(audio_flags),
            "visual_types":       list(visual_types_set),
            "audio_types":        list(audio_types_set),
        }
    }

    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)

    print(f"\n✅ Pipeline complete!")
    print(f"   Safe video  → {output_path}")
    print(f"   JSON report → {report_path}")
    print(f"   Visual flags: {len(video_flags)}")
    print(f"   Audio flags : {len(audio_flags)}")

    return report


# ─── Run it ──────────────────────────────────────────────────────────────────
report = run_pipeline()


🎥 STEP 1 / 4 — Visual detection & remediation
📹 Video: 1280x720 @ 30.0 fps | 315 frames


Processing frames: 100%|██████████| 315/315 [00:30<00:00, 10.20frame/s]


✅ Frame processing complete | total_frames=315 | blurred_frames=214 | unique_visual_flags=214

🎤 STEP 2 / 4 — Audio extraction

🔍 STEP 3 / 4 — Speech transcription & harmful speech detection
Detected language: English


100%|██████████| 1045/1045 [00:03<00:00, 347.64frames/s]


   Transcribed 23 word segment(s) | 2 audio flag(s)

🔗 STEP 4 / 4 — Merging remediated video & audio

✅ Pipeline complete!
   Safe video  → output_safe.mp4
   JSON report → remediation_report.json
   Visual flags: 214
   Audio flags : 2


## 9. 📄 View Detection Report

In [72]:
from IPython.display import JSON, display

print("=" * 60)
print("REMEDIATION SUMMARY")
print("=" * 60)
print(f"  Total visual flags : {report['summary']['total_visual_flags']}")
print(f"  Visual types found : {report['summary']['visual_types']}")
print(f"  Total audio flags  : {report['summary']['total_audio_flags']}")
print(f"  Audio types found  : {report['summary']['audio_types']}")
print()

if report['video_flags']:
    print("── Visual Flags ──")
    for f in report['video_flags'][:10]:
        visual_label = ", ".join(f.get("types", [f.get("type", "unknown")]))
        print(f"  [{f['timestamp']:.2f}s] {visual_label:20s} conf={f['confidence']:.2f}  → {f['action']}")
    if len(report['video_flags']) > 10:
        print(f"  ... and {len(report['video_flags'])-10} more")

if report['audio_flags']:
    print("\n── Audio Flags ──")
    for f in report['audio_flags']:
        print(f"  [{f['start']:.2f}s – {f['end']:.2f}s]  \"{f['text']}\"  reason={f['reason']}  → {f['action']}")

print()
print("Full JSON report:")
display(JSON(report))

REMEDIATION SUMMARY
  Total visual flags : 214
  Visual types found : ['weapon']
  Total audio flags  : 2
  Audio types found  : ['profanity']

── Visual Flags ──
  [3.36s] weapon               conf=0.49  → blurred_frame
  [3.40s] weapon               conf=0.49  → blurred_frame
  [3.43s] weapon               conf=0.49  → blurred_frame
  [3.46s] weapon               conf=0.60  → blurred_frame
  [3.50s] weapon               conf=0.60  → blurred_frame
  [3.53s] weapon               conf=0.60  → blurred_frame
  [3.56s] weapon               conf=0.60  → blurred_frame
  [3.60s] weapon               conf=0.60  → blurred_frame
  [3.63s] weapon               conf=0.60  → blurred_frame
  [3.66s] weapon               conf=0.60  → blurred_frame
  ... and 204 more

── Audio Flags ──
  [5.62s – 7.06s]  "fuck fuck fuck fuck fuck"  reason=profanity  → beeped
  [8.04s – 8.60s]  "fuck."  reason=profanity  → beeped

Full JSON report:


<IPython.core.display.JSON object>

## 10. 🖥️ Preview Remediated Video (Jupyter)

In [73]:
from IPython.display import Video as IPVideo
import os

output_path = CONFIG["output_video"]
if os.path.exists(output_path):
    display(IPVideo(output_path, embed=True, width=640))
else:
    print(f"Output video not found at: {output_path}")

---
## 📌 Notes & Customization

| What to change | Where |
|---|---|
| Input/output video paths | `CONFIG` dict in Cell 2 |
| YOLO model (use custom `best.pt`) | `CONFIG['yolo_model']` |
| Add/remove harmful object classes | `CONFIG['harmful_visual_classes']` |
| Whisper model size (accuracy vs speed) | `CONFIG['whisper_model']` |
| Switch mute ↔ beep | `CONFIG['use_beep']` |
| Extra profanity words | `CONFIG['extra_bad_words']` |
| Toxicity threshold | `TOXICITY_THRESHOLD` in Cell 5 |
| Frame analysis frequency | `CONFIG['frame_interval_sec']` |

### Custom YOLO Model
If you have a custom-trained `best.pt` model for cigarettes, guns, drugs etc., set:
```python
CONFIG['yolo_model'] = '/path/to/best.pt'
```

### Supported Output
- `output_safe.mp4` — remediated video with blurred visuals and censored audio
- `remediation_report.json` — structured detection log with timestamps